In [ ]:
import sys
from pathlib import Path
for _root in [Path.cwd(), *Path.cwd().parents]:
    if (_root / "paths.py").exists():
        sys.path.insert(0, str(_root))
        break
else:
    raise RuntimeError(
        "Could not find Llama-70B project root (paths.py). Run Jupyter with cwd project root or notebooks/."
    )
import paths


In [1]:
import os

os.environ["HF_HOME"] = "/orcd/compute/mghassem/001/gobi1/huggingface"
os.environ["TRANSFORMERS_CACHE"] = "/orcd/compute/mghassem/001/gobi1/huggingface"


In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "meta-llama/Llama-3.1-70B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    use_fast=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,   # recommended on A100/H100
    device_map="auto"
)


/home/yuexing/miniconda/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|█| 723/723 [02:31<00:00,  4.77it/s, Materializing param=model.norm.wei


In [3]:
prompt = "Give me a short introduction to large language model."
messages = [
    {"role": "system", "content": "You are Llama. You are a helpful medical assistant."},
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=2048,
    do_sample=False
)
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


In [4]:
import pandas as pd 
import re
import torch
import os

# Load data
df = pd.read_csv(paths.DATA / "Qwen14B_annotated_MedPAIR_relevancy.csv")
print("Columns in dataset:")
print(df.columns.tolist())

# Function to extract answer letter using multiple patterns
def extract_answer_letter(text):
    if pd.isna(text) or not text:
        return None
    
    # Try different patterns to extract the answer letter
    patterns = [
        r"Answer:\s*([A-J])",             # "Answer: A"
        r"Answer is\s*([A-J])",           # "Answer is A"
        r"answer is\s*([A-J])",           # "answer is A"
        r"The answer is\s*([A-J])",       # "The answer is A"
        r"the answer is\s*([A-J])",       # "the answer is A"
        r"Option\s*([A-J])",              # "Option A"
        r"option\s*([A-J])",              # "option A"
        r"My answer is\s*([A-J])",        # "My answer is A"
        r"(\n|^)([A-J])\.?\s*$",          # "A." or just "A" at end or newline
        r"select option\s*([A-J])",       # "select option A"
        r"I select\s*([A-J])",            # "I select A"
        r"I choose\s*([A-J])",            # "I choose A"
    ]
    
    for pattern in patterns:
        match = re.search(pattern, text)
        if match:
            # Some patterns have the letter in group 1, others in group 2
            return match.group(1) if len(match.groups()) == 1 else match.group(2)
    
    # If no match found, check if there's a single letter at the end
    words = text.strip().split()
    if words and len(words[-1]) == 1 and words[-1].isalpha() and words[-1].upper() in "ABCDEFGHIJ":
        return words[-1].upper()
    
    return None


# At the beginning, before the loop
progress_file = paths.PREDICTIONS / "[SR]_Llama70B_predictions_on_14B_progress.csv"

# Check if progress file exists and load it
if os.path.exists(progress_file):
    existing_results = pd.read_csv(progress_file)
    # Extract processed row indices from QA_ID (format: "Merge Q123")
    processed_indices = set()
    for qa_id in existing_results['QA_ID']:
        # Extract number from "Merge Q123" -> 123, then convert to 0-indexed (122)
        idx = int(qa_id.split('Q')[1]) - 1
        processed_indices.add(idx)
    
    results = existing_results.to_dict('records')
    print(f"Found {len(processed_indices)} already processed rows. Resuming...")
else:
    processed_indices = set()
    results = []
    print("Starting from scratch...")


# Loop through the dataset
total_rows = len(df)
print(f"Processing {total_rows} rows...")

for idx, row in df.head(total_rows).iterrows():
    # Skip if already processed
    if idx in processed_indices:
        print(f"Skipping row {idx+1}/{total_rows} (already processed)...")
        continue
    
    print(f"Processing row {idx+1}/{total_rows}...")
    try:
        context_text = row["Qwen14B_High_Relevance"]
        question = row["question_options_x"]
        
        # Improved prompt with clearer instructions
        prompt = (
            "You are a clinical reasoning assistant. You will receive a patient case summary "
            "and a multiple-choice question.\n\n"
            f"{context_text}\n\n"
            f"{question}\n\n"
            "Please select the single most appropriate answer. Respond only in the following format:\n\n"
            "Answer: <LETTER>"
        )
    
        # Use chat template format (like your working example)
        messages = [
            {"role": "system", "content": "You are Llama. You are a helpful medical assistant."},
            {"role": "user", "content": prompt}
        ]
        
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        
        model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
        
        # Generate prediction
        with torch.no_grad():
            generated_ids = model.generate(
                **model_inputs,
                max_new_tokens=100,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )
        
        # Extract only the generated part (not the input)
        generated_ids = [
            output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
        ]
        
        # Decode the response
        raw_response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
        
        # Extract the answer letter
        extracted_answer = extract_answer_letter(raw_response)
        
        
        # If still no answer found, log more details for debugging
        if extracted_answer is None:
            print(f"⚠️ Could not extract answer from response for row {idx+1}:")
            print(f"Response: {raw_response[:100]}...")
        
        # Create result entry
        qa_id = f"Merge Q{idx + 1}"
        result_entry = {
            "QA_ID": qa_id,
            "Origin": row.get("Origin", ""),
            "data_source": row.get("data_source_corr", ""),
            "Raw_Response": raw_response,
            "Extracted_Answer": extracted_answer
        }
        
        results.append(result_entry)
        processed_indices.add(idx)
        print(f"✅ Processed {qa_id}: Answer = {extracted_answer}")
        
        # Save progress every 10 items (increased frequency for safety)
        if (idx + 1) % 10 == 0:
            temp_df = pd.DataFrame(results)
            temp_df.to_csv(progress_file, index=False)
            print(f"Saved progress to CSV after {idx+1} items")

    except Exception as e:
        print(f"❌ Error on row {idx}: {str(e)}")
        # Still try to save the entry with error info
        qa_id = f"Merge Q{idx + 1}"
        results.append({
            "QA_ID": qa_id,
            "Origin": row.get("Origin", ""),
            "data_source": row.get("data_source_corr", ""),
            "Raw_Response": f"ERROR: {str(e)}",
            "Extracted_Answer": None
        })
        processed_indices.add(idx)

# Save final results
output_df = pd.DataFrame(results)
output_file = paths.PREDICTIONS / "[SR]_Llama70B_predictions_on_14B.csv"
output_df.to_csv(output_file, index=False)
print(f"Saved all predictions to {output_file}")

Columns in dataset:
['Origin', 'data_source_df3', 'Patient_Profile', 'Low+Irr', 'High', 'question_options_x', 'answer_corr', 'ID', 'centaur_question', 'sentence_number', 'answer', 'data_source', 'step1_excerpts', 'question_options_y', 'step1_sentences', 'sentence_1', 'sentence_2', 'sentence_3', 'sentence_4', 'sentence_5', 'sentence_6', 'sentence_7', 'sentence_8', 'sentence_9', 'sentence_10', 'sentence_11', 'sentence_12', 'sentence_13', 'sentence_14', 'sentence_15', 'sentence_16', 'sentence_17', 'sentence_18', 'sentence_19', 'sentence_20', 'sentence_21', 'Qwen14B_answer', 'Qwen14B_raw_response', 'q1', 'q2', 'q3', 'q4', 'q5', 'q6', 'q7', 'q8', 'q9', 'q10', 'q11', 'q12', 'q13', 'q14', 'q15', 'q16', 'q17', 'q18', 'q19', 'q20', 'label_21', 'Qwen14B_High_Relevance']
Found 370 already processed rows. Resuming...
Processing 1303 rows...
Skipping row 1/1303 (already processed)...
Skipping row 2/1303 (already processed)...
Skipping row 3/1303 (already processed)...
Skipping row 4/1303 (already p

✅ Processed Merge Q371: Answer = D
Processing row 372/1303...
✅ Processed Merge Q372: Answer = D
Processing row 373/1303...
✅ Processed Merge Q373: Answer = B
Processing row 374/1303...
✅ Processed Merge Q374: Answer = C
Processing row 375/1303...
✅ Processed Merge Q375: Answer = D
Processing row 376/1303...
✅ Processed Merge Q376: Answer = A
Processing row 377/1303...
✅ Processed Merge Q377: Answer = C
Processing row 378/1303...
✅ Processed Merge Q378: Answer = C
Processing row 379/1303...
✅ Processed Merge Q379: Answer = A
Processing row 380/1303...
✅ Processed Merge Q380: Answer = E
Saved progress to CSV after 380 items
Processing row 381/1303...
⚠️ Could not extract answer from response for row 381:
Response: There is no patient case summary provided. Please provide the patient case summary so I can assist y...
✅ Processed Merge Q381: Answer = None
Processing row 382/1303...
✅ Processed Merge Q382: Answer = A
Processing row 383/1303...
✅ Processed Merge Q383: Answer = C
Processing 

✅ Processed Merge Q490: Answer = C
Saved progress to CSV after 490 items
Processing row 491/1303...
✅ Processed Merge Q491: Answer = A
Processing row 492/1303...
✅ Processed Merge Q492: Answer = C
Processing row 493/1303...
✅ Processed Merge Q493: Answer = B
Processing row 494/1303...
✅ Processed Merge Q494: Answer = C
Processing row 495/1303...
✅ Processed Merge Q495: Answer = A
Processing row 496/1303...
✅ Processed Merge Q496: Answer = B
Processing row 497/1303...
✅ Processed Merge Q497: Answer = A
Processing row 498/1303...
✅ Processed Merge Q498: Answer = C
Processing row 499/1303...
⚠️ Could not extract answer from response for row 499:
Response: I need the patient case summary to provide an accurate answer....
✅ Processed Merge Q499: Answer = None
Processing row 500/1303...
✅ Processed Merge Q500: Answer = C
Saved progress to CSV after 500 items
Processing row 501/1303...
✅ Processed Merge Q501: Answer = C
Processing row 502/1303...
✅ Processed Merge Q502: Answer = H
Processing 

✅ Processed Merge Q604: Answer = B
Processing row 605/1303...
✅ Processed Merge Q605: Answer = H
Processing row 606/1303...
✅ Processed Merge Q606: Answer = A
Processing row 607/1303...
✅ Processed Merge Q607: Answer = A
Processing row 608/1303...
✅ Processed Merge Q608: Answer = D
Processing row 609/1303...
✅ Processed Merge Q609: Answer = I
Processing row 610/1303...
⚠️ Could not extract answer from response for row 610:
Response: I'm happy to help. However, I don't see Figure A. Please provide the figure or describe it to me so ...
✅ Processed Merge Q610: Answer = None
Saved progress to CSV after 610 items
Processing row 611/1303...
✅ Processed Merge Q611: Answer = C
Processing row 612/1303...
✅ Processed Merge Q612: Answer = C
Processing row 613/1303...
✅ Processed Merge Q613: Answer = A
Processing row 614/1303...
✅ Processed Merge Q614: Answer = A
Processing row 615/1303...
✅ Processed Merge Q615: Answer = D
Processing row 616/1303...
✅ Processed Merge Q616: Answer = A
Processing 

✅ Processed Merge Q719: Answer = D
Processing row 720/1303...
✅ Processed Merge Q720: Answer = C
Saved progress to CSV after 720 items
Processing row 721/1303...
✅ Processed Merge Q721: Answer = G
Processing row 722/1303...
✅ Processed Merge Q722: Answer = B
Processing row 723/1303...
✅ Processed Merge Q723: Answer = A
Processing row 724/1303...
✅ Processed Merge Q724: Answer = A
Processing row 725/1303...
✅ Processed Merge Q725: Answer = C
Processing row 726/1303...
✅ Processed Merge Q726: Answer = C
Processing row 727/1303...
✅ Processed Merge Q727: Answer = A
Processing row 728/1303...
✅ Processed Merge Q728: Answer = F
Processing row 729/1303...
✅ Processed Merge Q729: Answer = D
Processing row 730/1303...
✅ Processed Merge Q730: Answer = C
Saved progress to CSV after 730 items
Processing row 731/1303...
✅ Processed Merge Q731: Answer = C
Processing row 732/1303...
✅ Processed Merge Q732: Answer = A
Processing row 733/1303...
✅ Processed Merge Q733: Answer = B
Processing row 734/13

✅ Processed Merge Q844: Answer = D
Processing row 845/1303...
✅ Processed Merge Q845: Answer = B
Processing row 846/1303...
✅ Processed Merge Q846: Answer = B
Processing row 847/1303...
✅ Processed Merge Q847: Answer = C
Processing row 848/1303...
✅ Processed Merge Q848: Answer = A
Processing row 849/1303...
✅ Processed Merge Q849: Answer = C
Processing row 850/1303...
✅ Processed Merge Q850: Answer = A
Saved progress to CSV after 850 items
Processing row 851/1303...
✅ Processed Merge Q851: Answer = C
Processing row 852/1303...
✅ Processed Merge Q852: Answer = A
Processing row 853/1303...
✅ Processed Merge Q853: Answer = D
Processing row 854/1303...
✅ Processed Merge Q854: Answer = A
Processing row 855/1303...
✅ Processed Merge Q855: Answer = B
Processing row 856/1303...
✅ Processed Merge Q856: Answer = B
Processing row 857/1303...
✅ Processed Merge Q857: Answer = B
Processing row 858/1303...
✅ Processed Merge Q858: Answer = C
Processing row 859/1303...
✅ Processed Merge Q859: Answer =

✅ Processed Merge Q964: Answer = D
Processing row 965/1303...
✅ Processed Merge Q965: Answer = F
Processing row 966/1303...
✅ Processed Merge Q966: Answer = C
Processing row 967/1303...
✅ Processed Merge Q967: Answer = B
Processing row 968/1303...
✅ Processed Merge Q968: Answer = A
Processing row 969/1303...
✅ Processed Merge Q969: Answer = C
Processing row 970/1303...
✅ Processed Merge Q970: Answer = E
Saved progress to CSV after 970 items
Processing row 971/1303...
✅ Processed Merge Q971: Answer = E
Processing row 972/1303...
✅ Processed Merge Q972: Answer = D
Processing row 973/1303...
✅ Processed Merge Q973: Answer = D
Processing row 974/1303...
✅ Processed Merge Q974: Answer = C
Processing row 975/1303...
✅ Processed Merge Q975: Answer = D
Processing row 976/1303...
✅ Processed Merge Q976: Answer = D
Processing row 977/1303...
✅ Processed Merge Q977: Answer = D
Processing row 978/1303...
✅ Processed Merge Q978: Answer = A
Processing row 979/1303...
✅ Processed Merge Q979: Answer =

✅ Processed Merge Q1079: Answer = B
Processing row 1080/1303...
✅ Processed Merge Q1080: Answer = D
Saved progress to CSV after 1080 items
Processing row 1081/1303...
✅ Processed Merge Q1081: Answer = C
Processing row 1082/1303...
✅ Processed Merge Q1082: Answer = C
Processing row 1083/1303...
✅ Processed Merge Q1083: Answer = J
Processing row 1084/1303...
✅ Processed Merge Q1084: Answer = B
Processing row 1085/1303...
✅ Processed Merge Q1085: Answer = C
Processing row 1086/1303...
⚠️ Could not extract answer from response for row 1086:
Response: There is not enough information to make a decision....
✅ Processed Merge Q1086: Answer = None
Processing row 1087/1303...
✅ Processed Merge Q1087: Answer = A
Processing row 1088/1303...
✅ Processed Merge Q1088: Answer = F
Processing row 1089/1303...
✅ Processed Merge Q1089: Answer = C
Processing row 1090/1303...
✅ Processed Merge Q1090: Answer = C
Saved progress to CSV after 1090 items
Processing row 1091/1303...
✅ Processed Merge Q1091: Answe

✅ Processed Merge Q1187: Answer = D
Processing row 1188/1303...
✅ Processed Merge Q1188: Answer = A
Processing row 1189/1303...
✅ Processed Merge Q1189: Answer = A
Processing row 1190/1303...
✅ Processed Merge Q1190: Answer = D
Saved progress to CSV after 1190 items
Processing row 1191/1303...
✅ Processed Merge Q1191: Answer = B
Processing row 1192/1303...
✅ Processed Merge Q1192: Answer = C
Processing row 1193/1303...
✅ Processed Merge Q1193: Answer = C
Processing row 1194/1303...
✅ Processed Merge Q1194: Answer = C
Processing row 1195/1303...
✅ Processed Merge Q1195: Answer = B
Processing row 1196/1303...
✅ Processed Merge Q1196: Answer = C
Processing row 1197/1303...
✅ Processed Merge Q1197: Answer = B
Processing row 1198/1303...
✅ Processed Merge Q1198: Answer = C
Processing row 1199/1303...
✅ Processed Merge Q1199: Answer = A
Processing row 1200/1303...
✅ Processed Merge Q1200: Answer = C
Saved progress to CSV after 1200 items
Processing row 1201/1303...
✅ Processed Merge Q1201: A

✅ Processed Merge Q1299: Answer = D
Processing row 1300/1303...
✅ Processed Merge Q1300: Answer = B
Saved progress to CSV after 1300 items
Processing row 1301/1303...
✅ Processed Merge Q1301: Answer = B
Processing row 1302/1303...
✅ Processed Merge Q1302: Answer = C
Processing row 1303/1303...
⚠️ Could not extract answer from response for row 1303:
Response: There is no patient case summary provided. Please provide the patient case summary so I can assist y...
✅ Processed Merge Q1303: Answer = None
Saved all predictions to [SR]_Llama70B_predictions_on_14B.csv


In [11]:
import pandas as pd
import numpy as np
from scipy import stats

# Load the data
output_df = pd.read_csv(paths.PREDICTIONS / "[SR]_Llama70B_predictions_on_14B.csv")

# Calculate accuracy
output_df['match'] = (output_df['Extracted_Answer'] == output_df['answer_corr'])

# Overall accuracy statistics
accuracy = output_df['match'][:1300].mean()
std_dev = output_df['match'].std()
n = 1300
se = std_dev / np.sqrt(n)
ci_95 = stats.t.interval(0.95, n-1, loc=accuracy, scale=se)

print("=" * 60)
print("OVERALL ACCURACY ANALYSIS")
print("=" * 60)
print(f"Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Standard Deviation: {std_dev:.4f}")
print(f"95% Confidence Interval: [{ci_95[0]:.4f}, {ci_95[1]:.4f}]")
print(f"95% CI (percentage): [{ci_95[0]*100:.2f}%, {ci_95[1]*100:.2f}%]")
print(f"Sample Size: {n}")
print("=" * 60)
print()

# Analysis by category
if 'data_source_corr' in output_df.columns:
    analysis_df = output_df.copy()
elif 'data_source_corr' in df.columns:
    analysis_df = output_df.copy()
    analysis_df['data_source_corr'] = output_df['data_source_corr']
else:
    print("Warning: 'data_source_corr' column not found in either dataframe")
    analysis_df = output_df.copy()

# Category-wise analysis
if 'data_source_corr' in analysis_df.columns:
    category_stats = analysis_df.groupby('data_source_corr')['match'].agg([
        ('Count', 'count'),
        ('Mean_Accuracy', 'mean'),
        ('Std_Dev', 'std'),
        ('SE', lambda x: x.std() / np.sqrt(len(x)))
    ]).reset_index()
    
    ci_lower = []
    ci_upper = []
    
    for idx, row in category_stats.iterrows():
        n_cat = row['Count']
        mean_cat = row['Mean_Accuracy']
        se_cat = row['SE']
        
        if n_cat > 1:
            ci = stats.t.interval(0.95, n_cat-1, loc=mean_cat, scale=se_cat)
            ci_lower.append(ci[0])
            ci_upper.append(ci[1])
        else:
            ci_lower.append(np.nan)
            ci_upper.append(np.nan)
    
    category_stats['CI_95_Lower'] = ci_lower
    category_stats['CI_95_Upper'] = ci_upper
    category_stats['Mean_Accuracy_%'] = category_stats['Mean_Accuracy'] * 100
    category_stats['Std_Dev_%'] = category_stats['Std_Dev'] * 100
    category_stats['CI_95_Lower_%'] = category_stats['CI_95_Lower'] * 100
    category_stats['CI_95_Upper_%'] = category_stats['CI_95_Upper'] * 100
    
    print("CATEGORY-WISE ACCURACY ANALYSIS")
    print("=" * 60)
    print(category_stats.to_string(index=False))
    print("=" * 60)
    print()

# Create summary statistics table
summary_table = pd.DataFrame({
    'Metric': ['Overall Accuracy', 'Standard Deviation', '95% CI Lower', '95% CI Upper', 'Sample Size'],
    'Value': [f"{accuracy:.4f} ({accuracy*100:.2f}%)", 
              f"{std_dev:.4f}", 
              f"{ci_95[0]:.4f} ({ci_95[0]*100:.2f}%)", 
              f"{ci_95[1]:.4f} ({ci_95[1]*100:.2f}%)", 
              n]
})

print("\nSUMMARY TABLE")
print("=" * 60)
print(summary_table.to_string(index=False))
print("=" * 60)

# Save the updated dataframe with the new column to CSV
output_df.to_csv(paths.PREDICTIONS / "Llama70B_predictions_with_answers.csv", index=False)
print("\nData saved to paths.PREDICTIONS / "Llama70B_predictions_with_answers.csv"")

OVERALL ACCURACY ANALYSIS
Accuracy: 0.4538 (45.38%)
Standard Deviation: 0.4980
95% Confidence Interval: [0.4268, 0.4809]
95% CI (percentage): [42.68%, 48.09%]
Sample Size: 1300

CATEGORY-WISE ACCURACY ANALYSIS
data_source_corr  Count  Mean_Accuracy  Std_Dev       SE  CI_95_Lower  CI_95_Upper  Mean_Accuracy_%  Std_Dev_%  CI_95_Lower_%  CI_95_Upper_%
            jama    582       0.458763 0.498725 0.020673     0.418160     0.499365        45.876289  49.872524      41.816031      49.936546
      medbullets    207       0.565217 0.496930 0.034539     0.497122     0.633313        56.521739  49.693021      49.712207      63.331272
        medxpert    318       0.191824 0.394356 0.022114     0.148314     0.235333        19.182390  39.435585      14.831440      23.533340
            mmlu    193       0.751295 0.433386 0.031196     0.689765     0.812826        75.129534  43.338647      68.976477      81.282590


SUMMARY TABLE
            Metric           Value
  Overall Accuracy 0.4538 (45.38%)